[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dpadillaor/Sem3_SI_ImageEmbedding/blob/main/Colab_Seminario3.ipynb)

# **IMPORTANTE**
## **CREA UNA COPIA DE ESTE NOTEBOOK EN TU ESPACIO DE GOOGLE DRIVE.**
### Deberás entregar la copia en la tarea del seminario.

# Seminario 3: De Píxeles a Significado Compartido
## Visión por Computador y Modelos Multimodales

---

### Contexto

En los dos seminarios anteriores aprendiste que el **texto se convierte en vectores** y que puedes medir similitud semántica con la **similitud coseno**. Hoy vas a descubrir que las **imágenes también se pueden convertir en vectores**... y que texto e imagen pueden vivir en el **mismo espacio vectorial**.

El coseno que usabas para comparar *'banco'* y *'financiero'* va a funcionar ahora para comparar una frase y una foto.

### Pregunta que abre este seminario

> *¿Puede una máquina entender que esta foto y esta frase dicen lo mismo?*

Al final de la sesión habrás construido tú mismo la respuesta.

### Objetivos de aprendizaje

Al finalizar esta práctica serás capaz de:
- Entender cómo una imagen se representa como datos numéricos
- Comprender la analogía entre tokenización de texto y *patchification* de imágenes
- Generar embeddings visuales y medir similitud entre imágenes
- Entender cómo CLIP conecta texto e imagen en un espacio compartido
- Construir un clasificador zero-shot y un buscador semántico de imágenes

### Estructura de la sesión (~2 horas)

| Parte | Tema | Tiempo |
|-------|------|--------|
| Setup | Instalación y carga de modelos | 5 min |
| Hook | El truco de magia: CLIP antes de explicarlo | 10 min |
| I | Imágenes como datos: píxeles y patchification | 20 min |
| II | Embeddings visuales | 20 min |
| III | El espacio compartido texto ↔ imagen | 25 min |
| IV | Zero-shot classification y búsqueda semántica | 20 min |
| V *(opcional)* | Segmentación guiada por texto con SAM | 15 min |

---
## Configuración del entorno

Ejecuta esta celda primero. Puede tardar un par de minutos la primera vez. Este bloque instalará todas las dependencias necesarias para realizar este seminario.

In [ ]:
%%capture
!pip install transformers torch torchvision Pillow scikit-learn matplotlib seaborn

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully")

A continuación cargaremos el modelo que utilizaremos para esta sesión. En este caso usaremos **CLIP** un modelo dual de OpenAI que permite hacer *embeddings de imagenes y de texto*. Para cada caso, se usan mecanismos distintos que veremos más adeante en el seminario. 

Una vez cargado el modelo, para hacer la practica más sencilla, creamos tres funciones helper:
1. Función para **normalizar vectores**. Recordemos que para hacer cosine similarity, los vectores deben estar normalizados.
2. **Encoder de imagen**. El modelo utiliza el mecanismo de codificación de imagenes para conseguir un embedding de la imagen. El output se devuelve normalizado.
3. **Encoder de texto**. Devuelve un embedding de texto como en los seminarios anteriores. Output normalizado.

Además podéis ver como estamos usando el modelo en **inferencia**. Esto quiere decir que el modelo no cambia los parametros internos como en el entrenamiento. Usamos la red cya entrenada. De ahí el uso de `torch.no_grad()`.

In [ ]:
# Load CLIP model — this is the star of today's session
from transformers import CLIPProcessor, CLIPModel
import torch

clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("CLIP loaded successfully")

# ---- Helper: normalize a vector or batch of vectors ----
def normalize(v):
    """L2-normalize rows of a numpy array."""
    norms = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / (norms + 1e-8)

# ---- Helper: encode images with CLIP ----
def encode_images(images):
    """Encode a list of PIL images. Returns normalized numpy array (N, 512)."""
    inputs = clip_processor(images=images, return_tensors="pt", padding=True)
    with torch.no_grad():
        out = clip_model.get_image_features(**inputs)
    features = out if isinstance(out, torch.Tensor) else out.pooler_output
    return normalize(features.detach().numpy())

# ---- Helper: encode texts with CLIP ----
def encode_texts(texts):
    """Encode a list of strings. Returns normalized numpy array (N, 512)."""
    inputs = clip_processor(text=texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        out = clip_model.get_text_features(**inputs)
    features = out if isinstance(out, torch.Tensor) else out.pooler_output
    return normalize(features.detach().numpy())

Para los ejercicios de este seminario necesitamos un conjunto de imágenes con las que experimentar. Usaremos **CIFAR-10**, un dataset clásico de visión por computador creado por el Canadian Institute For Advanced Research.

CIFAR-10 contiene 60.000 imágenes de **32×32 píxeles** organizadas en **10 categorías**: *aviones*, *coches*, *pájaros*, *gatos*, *ciervos*, *perros*, *ranas*, *caballos*, *barcos* y *camiones*. Es pequeño y manejable, pero lo suficientemente variado como para que los experimentos sean interesantes.

Al ejecutar la celda verás que torchvision lo descarga automáticamente. A continuación:
1. Cargamos el conjunto de test (10.000 imágenes) — nunca se ha usado para entrenar nada.
2. Pre-seleccionamos 10 imágenes por clase (100 en total) para tener acceso rápido durante los ejercicios.
3. Visualizamos una imagen por clase para familiarizarnos con el dataset.

> Fíjate en la resolución: 32×32 píxeles es muy pequeña. Aun así, veremos que CLIP es capaz de entender su contenido.   Esto nos dice algo interesante

In [ ]:
# Load CIFAR-10 test set (10,000 images, 32x32, 10 classes)
import torchvision.datasets as datasets

cifar = datasets.CIFAR10(root='./data', train=False, download=True)

CLASS_NAMES = cifar.classes
print(f"CIFAR-10 loaded: {len(cifar)} images")
print(f"Classes: {CLASS_NAMES}")
print(f"Image type: {type(cifar[0][0])}, size: {cifar[0][0].size}")

# Pre-select 10 images per class for quick access (100 total)
class_indices = {c: [] for c in range(10)}
for idx, (_, label) in enumerate(cifar):
    if len(class_indices[label]) < 10:
        class_indices[label].append(idx)
    if all(len(v) == 10 for v in class_indices.values()):
        break

# Quick display: one image per class
fig, axes = plt.subplots(1, 10, figsize=(18, 2.5))
for c, ax in enumerate(axes):
    idx = class_indices[c][0]
    ax.imshow(cifar[idx][0])
    ax.set_title(CLASS_NAMES[c], fontsize=9)
    ax.axis('off')
plt.suptitle("CIFAR-10: one image per class", fontweight='bold')
plt.tight_layout()
plt.show()

---
## El truco de magia

Antes de explicar nada, vamos a ver CLIP en acción.

A continuación verás **4 imágenes** y **4 frases**. El modelo va a decir qué frase describe mejor cada imagen — usando solo similitud coseno entre sus representaciones vectoriales.

> **Pregunta antes de ejecutar:** ¿Cómo creéis que un modelo puede saber que una foto y una frase están relacionadas? ¿Alguna idea?

In [ ]:
# GIVEN CODE — just run and observe

# Pick one image per class (dog, airplane, car, frog)
hook_classes = ['dog', 'airplane', 'automobile', 'frog']
hook_indices = [class_indices[CLASS_NAMES.index(c)][0] for c in hook_classes]
hook_images  = [cifar[i][0] for i in hook_indices]
hook_texts   = [
    "a photo of a dog",
    "a photo of an airplane",
    "a photo of a car",
    "a photo of a frog",
]

# Encode images and texts with CLIP
img_embs  = encode_images(hook_images)   # shape (4, 512)
text_embs = encode_texts(hook_texts)     # shape (4, 512)

# Cosine similarity matrix: texts (rows) x images (cols)
sim_matrix = cosine_similarity(text_embs, img_embs)  # shape (4, 4)
n = len(hook_images)

# --- Visualization: images as thumbnails below the heatmap ---
plt.figure(figsize=(8, 7))
plt.imshow(sim_matrix, vmin=0.1, vmax=0.35, cmap='RdYlGn')

# Text labels on y-axis
short_texts = [t.replace("a photo of ", "") for t in hook_texts]
plt.yticks(range(n), short_texts, fontsize=13)
plt.xticks([])

# Similarity values inside cells
for x in range(n):
    for y in range(n):
        plt.text(x, y, f"{sim_matrix[y, x]:.3f}",
                 ha="center", va="center", size=12,
                 color="black")

# Images as thumbnails below x-axis
for i, img in enumerate(hook_images):
    plt.imshow(img, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")

# Clean up spines
for side in ["left", "top", "right", "bottom"]:
    plt.gca().spines[side].set_visible(False)

plt.xlim([-0.5, n - 0.5])
plt.ylim([n + 0.5, -2])
plt.title("Similitud coseno CLIP: texto × imagen", size=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Verify diagonal is highest per row
print()
for i, text in enumerate(short_texts):
    is_max = sim_matrix[i].argmax() == i
    print(f"{text:12s}: {sim_matrix[i, i]:.3f}  {'✓' if is_max else '✗'}")

---
## Parte I: Imágenes como datos

### Fundamento teórico

Para un ordenador, una imagen es simplemente una **matriz de números**. Cada píxel tiene tres valores entre 0 y 255, uno por canal: **R**ojo, **V**erde, **A**zul (RGB).

Pero esta representación no tiene semántica: dos fotos de perros distintos son matrices completamente diferentes. Necesitamos un **encoder** que entienda el contenido visual, igual que el tokenizador + modelo de lenguaje entiende el texto.

Antes de llegar ahí, vamos a entender cómo se divide la imagen en unidades procesables — la equivalencia visual de la tokenización.

### La analogía fundamental

| Texto | Imagen |
|---|---|
| `"hola mundo"` | foto de un gato |
| tokens: `["hola", "mundo"]` | patches: `[parche₁, parche₂, ...]` |
| embedding por token | embedding por patch |
| `[CLS]` token resumen global | class token resumen global |


<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/Patchification.png" width="800">
</div>


### Ejemplo guiado: la imagen como datos numéricos
En este ejemplo vamos a ver cómo el ordenador "ve" realmente una imagen. No ve un perro — ve una matriz de números. Ejecuta la celda y observa:

- El shape (32, 32, 3) → alto × ancho × canales de color (R, G, B)
- Los valores de píxel van de 0 a 255
- Los tres canales separados: cada uno captura la intensidad de un color

>Esto es lo que recibe el modelo antes de hacer cualquier cosa inteligente. La magia empieza después.

In [ ]:
# GUIDED EXAMPLE — run and observe

# Load one CIFAR image
sample_img, sample_label = cifar[class_indices[CLASS_NAMES.index('dog')][0]]
img_array = np.array(sample_img)  # shape (32, 32, 3)

print(f"Image shape: {img_array.shape}  → (height, width, channels)")
print(f"Pixel values range: {img_array.min()} to {img_array.max()}")
print(f"Label: {CLASS_NAMES[sample_label]}")
print(f"\nTop-left 3x3 pixels (RGB values):")
print(img_array[:3, :3, :])

# Visualize the image and its three RGB channels
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(sample_img)
axes[0].set_title(f"Original ({CLASS_NAMES[sample_label]})", fontweight='bold')

channel_names = ['Red channel', 'Green channel', 'Blue channel']
cmaps = ['Reds', 'Greens', 'Blues']
for i, (ax, name, cmap) in enumerate(zip(axes[1:], channel_names, cmaps)):
    channel = img_array[:, :, i]
    ax.imshow(channel, cmap=cmap)
    ax.set_title(name)
    ax.set_xlabel(f"min={channel.min()}, max={channel.max()}")

for ax in axes:
    ax.axis('off')
axes[1].axis('on'); axes[1].set_xticks([]); axes[1].set_yticks([])
axes[2].axis('on'); axes[2].set_xticks([]); axes[2].set_yticks([])
axes[3].axis('on'); axes[3].set_xticks([]); axes[3].set_yticks([])

plt.suptitle('A dog image seen as numbers: three channels (R, G, B)',
             fontweight='bold')
plt.tight_layout()
plt.show()

### Ejercicio 1: Patchification — la tokenización visual (Dificultad: Baja)

Los modelos Vision Transformer (ViT) — los que usa CLIP internamente — dividen la imagen en **patches** (parches) de tamaño fijo, igual que el tokenizador divide el texto en tokens.

> **Pregunta antes de programar:** Una imagen es de 32×32 píxeles y cada patch mide 8×8, ¿cuántos patches (tokens visuales) tiene la imagen?

Completa el código para dividir la imagen en patches y visualizarlos en una cuadrícula.

In [ ]:
patch_size = 8  # each patch is 8x8 pixels
img_array = np.array(sample_img)  # shape (32, 32, 3)

# TODO: Calculate how many patches fit along each dimension
n_patches_per_side = ...  # <-- COMPLETE HERE (hint: 32 // 8)

print(f"Image size: 32x32 pixels")
print(f"Patch size: {patch_size}x{patch_size} pixels")
print(f"Patches per side: {n_patches_per_side}")
print(f"Total patches (tokens): {n_patches_per_side**2 if n_patches_per_side else '?'}")

# TODO: Extract all patches and visualize them in a grid
fig, axes = plt.subplots(n_patches_per_side, n_patches_per_side,
                         figsize=(8, 8)) if n_patches_per_side else (None, None)

if n_patches_per_side:
    for i in range(n_patches_per_side):
        for j in range(n_patches_per_side):
            # TODO: Extract the (i, j) patch from img_array
            patch = img_array[
                i*patch_size : ...,   # <-- COMPLETE rows
                j*patch_size : ...,   # <-- COMPLETE cols
                :                     # all channels
            ]
            axes[i][j].imshow(patch)
            axes[i][j].axis('off')

    plt.suptitle(
        f"Dog image divided into {n_patches_per_side**2} patches of {patch_size}×{patch_size}px\n"
        f"Each patch = one visual 'token'",
        fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

**Reflexiona:** Cada patch contiene información visual local — el cielo en un patch, un ala en otro, una rueda en otro. Igual que cada token de texto tiene información semántica local. El transformer aprende a relacionar todos los patches entre sí para entender la imagen completa.

<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/Plane_patching.png" width="800">
</div>

### Ejercicio: ¿Qué pasa si mezclamos los patches?

Los tokens de texto tienen un orden que importa: "el perro muerde al hombre" ≠ "el hombre muerde al perro". ¿Pasa lo mismo con los patches visuales?

En este ejercicio vas a **reordenar aleatoriamente** los patches de una imagen y reconstruirla. Verás que para un humano la imagen se vuelve irreconocible — pero el modelo ViT recibe exactamente los mismos patches, solo en distinto orden.

> **Reflexiona antes:** ¿Crees que CLIP daría el mismo embedding a la imagen original y a la versión con patches mezclados? ¿Por qué?

In [ ]:
# OPTIONAL EXERCISE — patch shuffling

patch_size = 8
img_array = np.array(sample_img)  # (32, 32, 3)
n = 32 // patch_size  # patches per side

# Extract all patches into a list
patches = []
for i in range(n):
    for j in range(n):
        patch = img_array[i*patch_size:(i+1)*patch_size,
                          j*patch_size:(j+1)*patch_size, :]
        patches.append(patch)

# Shuffle the patch list and reconstruct the image
import random
shuffled = patches.copy()
random.shuffle(shuffled)

# Reconstruct shuffled image
rows = []
for i in range(n):
    row = np.concatenate(shuffled[i*n:(i+1)*n], axis=1)
    rows.append(row)
shuffled_img = np.concatenate(rows, axis=0)

# Visualize original vs shuffled
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(img_array)
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')
axes[1].imshow(shuffled_img)
axes[1].set_title('Patches shuffled', fontweight='bold')
axes[1].axis('off')
plt.suptitle(f'Same {n*n} patches, different order', fontweight='bold')
plt.tight_layout()
plt.show()

# Encode both with CLIP and compare
from PIL import Image as PILImage
orig_emb     = encode_images([sample_img])
shuffled_pil = PILImage.fromarray(shuffled_img)
shuffled_emb = encode_images([shuffled_pil])

sim = float(orig_emb @ shuffled_emb.T)
print(f"Cosine similarity (original vs shuffled): {sim:.3f}")
print("→ Are they the same for CLIP?")

### Introduce aqui tu respuesta. ¿Qué piensas?

Respuesta:

---
## Parte II: Embeddings visuales

### Fundamento teórico

Igual que el modelo de lenguaje que usamos en el primer seminario convertía cada token en un vector de 384 dimensiones, el encoder visual de CLIP convierte una imagen en un **vector de 512 dimensiones**.

Este vector captura el **significado visual** de la imagen, su **semántica**: dos fotos de perros producen vectores cercanos, aunque sean perros diferentes. Una foto de un avión produce un vector lejano.

<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/Visual Encoder.png" width="600">
</div>

### Ejemplo guiado: embedding de una imagen

In [ ]:
# GUIDED EXAMPLE — run and observe

dog_img = cifar[class_indices[CLASS_NAMES.index('dog')][0]][0]

# Encode one image with CLIP
dog_emb = encode_images([dog_img])  # shape (1, 512)

print(f"Image embedding shape: {dog_emb.shape}")
print(f"Embedding dimension: {dog_emb.shape[1]}  (same as CLIP text embeddings)")
print(f"First 10 values: {dog_emb[0][:10].round(4)}")
print(f"Norm (after normalization): {np.linalg.norm(dog_emb[0]):.4f}")

### Ejercicio 2: Similitud coseno entre imágenes (Dificultad: Baja)

¡El mismo operador que usabas en el Seminario 2 para comparar frases funciona ahora con imágenes!

> **Predice antes de ejecutar:** ¿Qué similitud coseno esperas entre dos fotos de perros distintos? ¿Y entre una foto de perro y una de avión? Apunta tus predicciones.

In [ ]:
# Select 6 representative images: 2 dogs, 2 airplanes, 1 car, 1 frog
selected = {
    'dog_1':      cifar[class_indices[CLASS_NAMES.index('dog')][0]][0],
    'dog_2':      cifar[class_indices[CLASS_NAMES.index('dog')][1]][0],
    'airplane_1': cifar[class_indices[CLASS_NAMES.index('airplane')][0]][0],
    'airplane_2': cifar[class_indices[CLASS_NAMES.index('airplane')][1]][0],
    'car':        cifar[class_indices[CLASS_NAMES.index('automobile')][0]][0],
    'frog':       cifar[class_indices[CLASS_NAMES.index('frog')][0]][0],
}

# TODO: Encode all images using encode_images()
# Store a numpy array of shape (6, 512)
names = list(selected.keys())
images = list(selected.values())

all_image_embs = ...  # <-- COMPLETE HERE

# TODO: Compute the cosine similarity matrix (6x6)
sim_matrix = ...  # <-- COMPLETE HERE

# Visualization (given)
if sim_matrix is not None and not isinstance(sim_matrix, type(...)):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: show the 6 images
    for i, (name, img) in enumerate(selected.items()):
        ax = fig.add_axes([0.02 + i*0.08, 0.6, 0.07, 0.25])
        ax.imshow(img)
        ax.set_title(name, fontsize=7)
        ax.axis('off')
    axes[0].axis('off')

    # Right: heatmap
    sns.heatmap(
        sim_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
        xticklabels=names, yticklabels=names,
        vmin=0.5, vmax=1.0, ax=axes[1], linewidths=0.5
    )
    axes[1].set_title('Cosine similarity between image embeddings',
                      fontweight='bold')
    axes[1].tick_params(axis='x', rotation=30)
    plt.suptitle('Similar images → closer vectors in embedding space',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("\nKey observations:")
    print(f"  dog_1 vs dog_2:      {sim_matrix[0,1]:.3f}")
    print(f"  dog_1 vs airplane_1: {sim_matrix[0,2]:.3f}")
    print(f"  airplane_1 vs airplane_2: {sim_matrix[2,3]:.3f}")

### Ejercicio 3: PCA de embeddings visuales (Dificultad: Media)

Vamos a proyectar los embeddings de 50 imágenes (5 por clase) a 2D con PCA, igual que hacíais con palabras en el Seminario 2. ¿Se agruparán por categoría?

> **Pregunta:** ¿Qué clases esperáis que estén más próximas entre sí en el espacio visual?

In [ ]:
# Build a set of 5 images per class (50 total)
pca_images = []
pca_labels = []
for c in range(10):
    for idx in class_indices[c][:5]:
        pca_images.append(cifar[idx][0])
        pca_labels.append(CLASS_NAMES[c])

# TODO: Encode all 50 images
pca_embs = ...  # <-- COMPLETE HERE  shape: (50, 512)

# TODO: Apply PCA to reduce to 2 dimensions
pca = ...      # <-- COMPLETE HERE  hint: PCA(n_components=2)
embs_2d = ...  # <-- COMPLETE HERE  hint: pca.fit_transform(pca_embs)

# Visualization (given)
if embs_2d is not None and not isinstance(embs_2d, type(...)):
    palette = plt.cm.get_cmap('tab10', 10)
    fig, ax = plt.subplots(figsize=(11, 8))

    for i, (label, coords) in enumerate(zip(pca_labels, embs_2d)):
        c_idx = CLASS_NAMES.index(label)
        ax.scatter(coords[0], coords[1],
                   color=palette(c_idx), s=90, alpha=0.85,
                   edgecolors='white', linewidth=0.5, zorder=3)

    # Add class centroids as labels
    for c_idx, cls in enumerate(CLASS_NAMES):
        mask = [l == cls for l in pca_labels]
        cx = embs_2d[mask, 0].mean()
        cy = embs_2d[mask, 1].mean()
        ax.text(cx, cy, cls, fontsize=11, fontweight='bold',
                color=palette(c_idx),
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))

    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
    ax.set_title('PCA of CLIP image embeddings — 50 CIFAR-10 images (5 per class)',
                 fontweight='bold')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

    print(f"Total variance explained by 2 PCs: {pca.explained_variance_ratio_.sum():.1%}")
    print("\nWhat groupings do you see? Are similar classes close together?")

---
## Parte III: El espacio compartido texto ↔ imagen

### Fundamento teórico

Hasta ahora hemos usado el encoder de imágenes de CLIP. Pero CLIP tiene **dos encoders**:

- **Text Encoder**: texto → vector en espacio compartido (512D)
- **Image Encoder**: imagen → vector en **el mismo** espacio compartido (512D)

CLIP se entrenó con **400 millones de pares (imagen, descripción)** de Internet. El objetivo fue simple: hacer que el vector del texto y el vector de la imagen correspondiente estén cerca. Y hacer que pares incorrectos estén lejos.

El resultado: la **similitud coseno entre un texto y una imagen** funciona exactamente igual que entre dos textos.

---
## Parte II: Embeddings visuales

### Fundamento teórico

Igual que el modelo de lenguaje que usamos en el primer seminario convertía cada token en un vector de 384 dimensiones, el encoder visual de CLIP convierte una imagen en un **vector de 512 dimensiones**.

Este vector captura el **significado visual** de la imagen, su **semántica**: dos fotos de perros producen vectores cercanos, aunque sean perros diferentes. Una foto de un avión produce un vector lejano.

<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/Dual_Encoder.png" width="600">
</div>

### Ejemplo guiado: mismo espacio, dos modalidades

In [ ]:
# GUIDED EXAMPLE — run and observe

text = "a photo of a dog"
dog_img  = cifar[class_indices[CLASS_NAMES.index('dog')][0]][0]
plane_img = cifar[class_indices[CLASS_NAMES.index('airplane')][0]][0]

text_emb  = encode_texts([text])       # shape (1, 512)
dog_emb   = encode_images([dog_img])   # shape (1, 512)
plane_emb = encode_images([plane_img]) # shape (1, 512)

sim_text_dog   = cosine_similarity(text_emb, dog_emb)[0][0]
sim_text_plane = cosine_similarity(text_emb, plane_emb)[0][0]

print(f"Text embedding shape:  {text_emb.shape}")
print(f"Image embedding shape: {dog_emb.shape}")
print(f"Both are 512-dimensional → they live in the same space!")
print(f"\nCosine similarity:")
print(f"  '{text}' vs dog image:      {sim_text_dog:.4f}")
print(f"  '{text}' vs airplane image: {sim_text_plane:.4f}")

### Ejercicio 4: Matriz de similitud texto ↔ imagen (Dificultad: Media)

Vamos a construir una **matriz cruzada**: filas = imágenes, columnas = textos. Cada celda es la similitud coseno entre ese par.

> **Predice:** ¿Qué patrón esperas ver en la matriz? ¿Dónde estarán los valores más altos?

In [ ]:
# 5 images (one per class)
cross_classes  = ['dog', 'airplane', 'automobile', 'frog', 'horse']
cross_images   = [cifar[class_indices[CLASS_NAMES.index(c)][0]][0] for c in cross_classes]
cross_texts_en = [
    "a photo of a dog",
    "a photo of an airplane in the sky",
    "a photo of a car on the road",
    "a photo of a frog",
    "a photo of a horse",
]

# TODO: Encode all images
cross_img_embs = ...  # <-- COMPLETE HERE  shape: (5, 512)

# TODO: Encode all texts
cross_text_embs_en = ...  # <-- COMPLETE HERE  shape: (5, 512)

# TODO: Compute the cross-modal similarity matrix (texts x images)
cross_sim_en = ...  # <-- COMPLETE HERE  shape: (5, 5)

# Visualization (given) — texts on y-axis, images as thumbnails below x-axis
def plot_cross_similarity(sim_matrix, images, text_labels, title):
    n = len(images)
    short_labels = [t.replace("a photo of ", "") for t in text_labels]

    plt.figure(figsize=(8, 7))
    plt.imshow(sim_matrix, vmin=0.1, vmax=0.4, cmap='RdYlGn')
    plt.yticks(range(n), short_labels, fontsize=12)
    plt.xticks([])

    for x in range(n):
        for y in range(n):
            plt.text(x, y, f"{sim_matrix[y, x]:.3f}",
                     ha="center", va="center", size=11, color="black")

    for i, img in enumerate(images):
        plt.imshow(img, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")

    for side in ["left", "top", "right", "bottom"]:
        plt.gca().spines[side].set_visible(False)

    plt.xlim([-0.5, n - 0.5])
    plt.ylim([n + 0.5, -2])
    plt.title(title, size=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

if cross_sim_en is not None and not isinstance(cross_sim_en, type(...)):
    # Note: plot expects sim_matrix with texts as rows, images as cols
    plot_cross_similarity(
        cosine_similarity(cross_text_embs_en, cross_img_embs),
        cross_images, cross_texts_en,
        'Cross-modal similarity: images × English texts'
    )

### Ejercicio 5: ¿Funciona en castellano? (Dificultad: Baja)

CLIP se entrenó principalmente con textos en inglés. Sin embargo, en el Seminario 2 visteis que los embeddings multilingües pueden mapear idiomas distintos al mismo espacio.

> ¿Creéis que CLIP funcionará con queries en castellano? ¿Por qué sí o por qué no? Discutidlo antes de ejecutar.

In [ ]:
# Same images, same order — only the texts change
cross_texts_es = [
    "una foto de un perro",
    "un avión en el cielo",
    "un coche en la carretera",
    "una rana en una hoja",
    "un caballo en un campo",
]

# TODO: Encode the Spanish texts
cross_text_embs_es = ...  # <-- COMPLETE HERE

# TODO: Compute cross-modal similarity with Spanish texts
# (use the same cross_img_embs from Exercise 4 — images don't change!)
cross_sim_es = ...  # <-- COMPLETE HERE

# Visualization and comparison (given)
if cross_sim_es is not None and not isinstance(cross_sim_es, type(...)):
    plot_cross_similarity(
        cosine_similarity(cross_text_embs_es, cross_img_embs),
        cross_images, cross_texts_es,
        'Cross-modal similarity: images × Spanish texts'
    )

    # Compare diagonal values: English vs Spanish
    print("\nDiagonal comparison (correct text-image pairs):")
    print(f"{'Class':<12} {'English':>10} {'Spanish':>10} {'Difference':>12}")
    print('-' * 48)
    for i, cls in enumerate(cross_classes):
        en = cross_sim_en[i, i]
        es = cross_sim_es[i, i]
        print(f"{cls:<12} {en:>10.3f} {es:>10.3f} {es-en:>+12.3f}")

    print("\nWhy does it work in Spanish? Connect this with what you learnt in Seminar 2.")

---
## Parte IV: Zero-shot classification y búsqueda semántica

### Fundamento teórico

Una vez que texto e imagen viven en el mismo espacio, la **clasificación** es gratis: compara el embedding de la imagen con los embeddings de los nombres de cada clase y quédate con el más cercano. Sin entrenar nada. Sin ver ningún ejemplo etiquetado. Esto se llama **zero-shot classification**.

<br>
<div style="text-align: center;">
  <img src="https://raw.githubusercontent.com/dpadillaor/Sem3_SI_ImageEmbedding/main/img/Zero-Shot.png" width="900">
</div>

### Ejercicio 6: Zero-shot classification en CIFAR-10 (Dificultad: Media)

> **Apuesta:** Sin entrenar absolutamente nada, clasificando solo por similitud coseno con el nombre de la clase, ¿qué *accuracy* esperáis? La línea base aleatoria es el 10%. ¿Apostáis más de un 50%?

In [ ]:
# Text prompt for each class
class_prompts = [f"a photo of a {c}" for c in CLASS_NAMES]

# TODO: Encode all 10 class prompts
class_text_embs = ...  # <-- COMPLETE HERE  shape: (10, 512)

# Select 50 test images (5 per class)
test_images = []
test_labels = []
for c in range(10):
    for idx in class_indices[c][5:10]:  # Use 5 images not seen before
        test_images.append(cifar[idx][0])
        test_labels.append(c)

predicted_labels = []
for img in test_images:
    # TODO: Encode the image
    img_emb = ...  # <-- COMPLETE HERE  shape: (1, 512)

    # TODO: Compute similarity with all 10 class prompts
    sims = ...     # <-- COMPLETE HERE  shape: (1, 10)  or (10,)

    # TODO: Predicted class = index of highest similarity
    predicted_class = ...  # <-- COMPLETE HERE  hint: np.argmax(...)
    predicted_labels.append(predicted_class)

# TODO: Compute accuracy
accuracy = ...  # <-- COMPLETE HERE

print(f"Zero-shot classification accuracy on CIFAR-10: {accuracy:.1%}")
print(f"Random baseline: 10.0%")
print(f"CLIP improvement over random: {accuracy - 0.1:.1%}")

In [ ]:
# Confusion matrix (given)
from sklearn.metrics import confusion_matrix
import matplotlib.ticker as ticker

if accuracy is not None and not isinstance(accuracy, type(...)):
    cm = confusion_matrix(test_labels, predicted_labels)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=ax, linewidths=0.5
    )
    ax.set_xlabel('Predicted', fontweight='bold')
    ax.set_ylabel('True', fontweight='bold')
    ax.set_title(f'Confusion matrix — zero-shot accuracy: {accuracy:.1%}',
                 fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()
    print("\nWhich classes does CLIP confuse most often? Why?")

### Ejercicio 7: El prompt importa (Dificultad: Baja-Media)

La formulación del texto que usamos para representar cada clase afecta al resultado. Vamos a comparar distintas variantes.

> **Experimenta:** Añade tu propia variante de prompt. ¿Puedes mejorar la *accuracy* cambiando solo el texto?

In [ ]:
prompt_templates = [
    "{c}",                                   # just the class name
    "a photo of a {c}",                      # standard
    "a low resolution photo of a {c}",       # CIFAR-specific
    "a blurry small image of a {c}",
    "un {c}",                                # Spanish
    # TODO: Add your own template here!
    # "{c}",
]

print(f"{'Template':<40} {'Accuracy':>10}")
print('-' * 52)

for template in prompt_templates:
    prompts = [template.format(c=c) for c in CLASS_NAMES]

    # TODO: Encode these prompts
    tmpl_text_embs = ...  # <-- COMPLETE HERE

    # TODO: Classify test_images using this template and compute accuracy
    tmpl_preds = []
    for img in test_images:
        img_emb = ...  # <-- COMPLETE HERE
        sims    = ...  # <-- COMPLETE HERE
        tmpl_preds.append(...)  # <-- COMPLETE HERE

    tmpl_acc = ...  # <-- COMPLETE HERE
    print(f"{template:<40} {tmpl_acc:>10.1%}")

### Ejercicio 8: Buscador semántico de imágenes (Dificultad: Media)

Ahora construimos el **buscador semántico**: dado un texto en cualquier idioma, encontrar las imágenes más similares de un corpus.

Primero pre-computamos los embeddings de 500 imágenes. Luego buscamos con queries en castellano o inglés.

In [ ]:
# Pre-compute embeddings for a pool of 500 images
# (This takes ~2 minutes on CPU — run once and save)
import os

POOL_SIZE = 500
SAVE_PATH = './pool_embeddings.npy'

pool_images = [cifar[i][0] for i in range(POOL_SIZE)]
pool_labels = [cifar[i][1] for i in range(POOL_SIZE)]

if os.path.exists(SAVE_PATH):
    pool_embs = np.load(SAVE_PATH)
    print(f"Loaded pre-computed embeddings from {SAVE_PATH}")
else:
    print("Computing embeddings for 500 images (~2 min)...")
    pool_embs = []
    batch_size = 32
    for i in range(0, POOL_SIZE, batch_size):
        batch = pool_images[i:i+batch_size]
        batch_embs = encode_images(batch)
        pool_embs.append(batch_embs)
        print(f"  {min(i+batch_size, POOL_SIZE)}/{POOL_SIZE} images processed")
    pool_embs = np.vstack(pool_embs)
    np.save(SAVE_PATH, pool_embs)
    print(f"Saved to {SAVE_PATH}")

print(f"Pool embeddings shape: {pool_embs.shape}")

In [ ]:
def semantic_image_search(query_text, pool_embeddings, top_k=5):
    """
    Given a text query, return indices and scores of the top_k most similar images.

    Args:
        query_text: string (any language)
        pool_embeddings: numpy array (N, 512) — pre-computed image embeddings
        top_k: number of results to return

    Returns:
        top_indices: indices of the top_k most similar images
        top_scores: their cosine similarity scores
    """
    # TODO: Encode the query text
    query_emb = ...  # <-- COMPLETE HERE  shape: (1, 512)

    # TODO: Compute cosine similarity between query and all pool embeddings
    similarities = ...  # <-- COMPLETE HERE  shape: (N,)  hint: flatten the result

    # TODO: Get indices of top_k highest similarities (descending order)
    top_indices = ...  # <-- COMPLETE HERE  hint: np.argsort(...)[-top_k:][::-1]

    return top_indices, similarities[top_indices]


def show_search_results(query, indices, scores, pool_images, pool_labels):
    """Display top search results as a grid."""
    fig, axes = plt.subplots(1, len(indices), figsize=(3*len(indices), 3.5))
    if len(indices) == 1:
        axes = [axes]
    for ax, idx, score in zip(axes, indices, scores):
        ax.imshow(pool_images[idx])
        ax.set_title(f"{CLASS_NAMES[pool_labels[idx]]}\nsim={score:.3f}",
                     fontsize=9)
        ax.axis('off')
    plt.suptitle(f'Query: "{query}"', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()


# Test with several queries
queries = [
    "un animal en la naturaleza",
    "a vehicle with wheels",
    "something that flies",
    "an animal that lives in water",
]

for query in queries:
    indices, scores = semantic_image_search(query, pool_embs, top_k=5)
    if indices is not None and not isinstance(indices, type(...)):
        show_search_results(query, indices, scores, pool_images, pool_labels)

In [ ]:
# --- YOUR TURN: Try your own queries ---
# Add at least 3 original queries. Try things that might fail or surprise you.
# Ideas: 'algo peligroso', 'un animal pequeño', 'un vehículo grande', 'paz'

my_queries = [
    # TODO: Add your queries here
    # "...",
    # "...",
    # "...",
]

for query in my_queries:
    indices, scores = semantic_image_search(query, pool_embs, top_k=5)
    if indices is not None and not isinstance(indices, type(...)):
        show_search_results(query, indices, scores, pool_images, pool_labels)

---
## Preguntas de evaluación

Responde a las siguientes preguntas en este notebook.

---

**Pregunta 1 — Patchification y tokenización**

Explica la analogía entre la tokenización del texto (Seminario 1) y la *patchification* de imágenes que has visto hoy. ¿Qué papel juega el *class token* en ambos casos? ¿Por qué crees que esta arquitectura compartida permite que texto e imagen vivan en el mismo espacio?

*Escribe tu respuesta aquí*

---

**Pregunta 2 — Espacio compartido y multilingüismo**

En el Ejercicio 5 comprobaste que CLIP funciona en castellano aunque se entrenó principalmente en inglés. Conecta este resultado con lo que aprendiste sobre embeddings multilingües en el Seminario 2. ¿Por qué la similitud coseno entre "un perro" y una foto de un perro es razonablemente alta?

*Escribe tu respuesta aquí*

---

**Pregunta 3 — El prompt importa**

En el Ejercicio 7 viste que la formulación del texto afecta a la *accuracy* del clasificador zero-shot. ¿Por qué "a low resolution photo of a {c}" puede funcionar mejor que solo "{c}" para CIFAR-10? ¿Qué nos dice esto sobre cómo CLIP aprendió a conectar texto e imagen?

*Escribe tu respuesta aquí*

---

**Pregunta 4 — Confusiones del clasificador**

Mira la matriz de confusión del Ejercicio 6. ¿Qué pares de clases confunde más CLIP? ¿Tiene sentido semántico? ¿Por qué crees que algunas confusiones son más probables que otras?

*Escribe tu respuesta aquí*

---

**Pregunta 5 — Aplicación real**

Imagina que una empresa de comercio electrónico tiene 2 millones de fotos de productos sin ninguna etiqueta. Con lo que has aprendido hoy, describe brevemente cómo construirías: (a) un buscador semántico que permita a los clientes buscar con frases en lenguaje natural, y (b) un sistema de categorización automática de productos. ¿Qué ventajas tiene este enfoque frente a un clasificador entrenado de forma supervisada?

*Escribe tu respuesta aquí*